# ROGII v2 -- Train & Submit

**Mode:**
- `MODE = 'train'` -- process 773 wells, train LGB+CB+Ridge stack, predict test, submit (~4-6h on Kaggle CPU)
- `MODE = 'predict'` -- load pre-trained models, predict test, submit (~2 min)

**Kaggle setup:**
- Attach competition dataset `rogii-wellbore-geology-prediction`
- For predict mode: attach models dataset containing `models_v2/` folder

In [ ]:
!pip install -q numba lightgbm catboost tqdm joblib 2>/dev/null

import os, shutil

ON_KAGGLE = os.path.exists('/kaggle/input')

if ON_KAGGLE:
    import subprocess
    REPO = '/kaggle/working/ROGII'
    # Always fresh clone to get latest code
    if os.path.exists(REPO):
        shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', 'https://github.com/ibouftini/ROGII.git', REPO], check=True)
else:
    REPO = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if not os.path.exists(os.path.join(REPO, 'v2')):
        REPO = os.getcwd()

# Verify v2 exists
assert os.path.exists(os.path.join(REPO, 'v2', 'config.py')), \
    f'v2/config.py not found in {REPO} -- push latest code to GitHub first'
print(f'REPO={REPO}, ON_KAGGLE={ON_KAGGLE}')

In [ ]:
import sys, os, importlib

V2_DIR = os.path.join(REPO, 'v2')
if V2_DIR not in sys.path:
    sys.path.insert(0, V2_DIR)

# Import config explicitly from v2/ to avoid Kaggle's own 'config' module
spec = importlib.util.spec_from_file_location('config', os.path.join(V2_DIR, 'config.py'))
cfg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cfg)

MODE = 'train'  # 'train' or 'predict'

if ON_KAGGLE:
    COMP_DIR   = '/kaggle/input/rogii-wellbore-geology-prediction'
    TRAIN_DIR  = f'{COMP_DIR}/train'
    TEST_DIR   = f'{COMP_DIR}/test'
    MODELS_DIR = '/kaggle/working/models_v2'
    # Debug: check what's actually in the dirs
    print('train dir contents:', os.listdir(TRAIN_DIR)[:5] if os.path.exists(TRAIN_DIR) else 'NOT FOUND')
    print('test dir contents:', os.listdir(TEST_DIR)[:5] if os.path.exists(TEST_DIR) else 'NOT FOUND')
else:
    TRAIN_DIR  = os.path.join(REPO, 'data', 'train')
    TEST_DIR   = os.path.join(REPO, 'data', 'test')
    MODELS_DIR = os.path.join(REPO, 'models_v2')
    COMP_DIR   = os.path.join(REPO, 'data')

# For predict-only: auto-detect models from attached dataset
if MODE == 'predict' and ON_KAGGLE:
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.pkl') for f in files) and 'v2' in root:
            MODELS_DIR = root
            break
    print(f'Models dir: {MODELS_DIR}')

# Kaggle speed overrides for training
if MODE == 'train':
    cfg.PF_N_SEEDS = 32
    cfg.PF_N_PARTICLES = 200
    cfg.NCC_STRIDE = 6
    cfg.N_JOBS = 4

cfg.MODELS_DIR = type(cfg.MODELS_DIR)(MODELS_DIR)
print(f'MODE={MODE}, MODELS_DIR={MODELS_DIR}')

In [ ]:
from rogii_v2.pipeline import run_pipeline

if MODE == 'train':
    run_pipeline(cfg, mode='train',
                 train_dir=TRAIN_DIR, test_dir=TEST_DIR,
                 models_dir=MODELS_DIR)

In [ ]:
import pandas as pd

preds = run_pipeline(cfg, mode='predict',
                     train_dir=TRAIN_DIR, test_dir=TEST_DIR,
                     models_dir=MODELS_DIR)

# Align with sample submission
sample = pd.read_csv(f'{COMP_DIR}/sample_submission.csv')
id_col, tvt_col = sample.columns[0], sample.columns[1]
preds.columns = [id_col, tvt_col]
fallback = float(preds[tvt_col].median())

sub = sample[[id_col]].merge(preds, on=id_col, how='left')
sub[tvt_col] = sub[tvt_col].fillna(fallback)

assert len(sub) == len(sample), f'Row mismatch: {len(sub)} vs {len(sample)}'
assert sub[tvt_col].isna().sum() == 0, 'NaN in submission'

print(f'Rows: {len(sub):,}  (expected {len(sample):,})')
print(sub[tvt_col].describe())

sub.to_csv('submission.csv', index=False)
print('submission.csv saved')